In [1]:
# !pip install ultralytics opencv-python numpy

In [2]:
import cv2
import numpy as np
from ultralytics import YOLO

print("OpenCV version:", cv2.__version__)

OpenCV version: 5.0.0


In [3]:
model = YOLO('yolov8n.pt')  # downloads weights automatically on first run
print(model.names)  # full list of COCO classes the model can detect

{0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted plant', 59: 'bed', 60: 'dining table', 61: 'toilet', 62: 'tv', 63: 'laptop', 64: 'mouse', 65: 'remote', 66: 'keyboard', 67: 'cell phone', 68: 'microw

In [4]:
fruit_classes = ['banana', 'apple', 'orange']

In [5]:
VIDEO_SOURCE = 'fruit_input.mp4'  # replace with your dataset video, or use 0 for webcam
cap = cv2.VideoCapture(VIDEO_SOURCE)

if not cap.isOpened():
    print("Could not open video source:", VIDEO_SOURCE)
else:
    print("Video source opened successfully.")

Could not open video source: fruit_input.mp4


In [6]:
count = 0
frame_num = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Run YOLO inference on the frame
    results = model(frame, verbose=False)[0]

    # Draw bounding boxes and labels for detected fruits
    for box in results.boxes:
        cls_id = int(box.cls[0])
        label = model.names[cls_id]
        conf = float(box.conf[0])
        x1, y1, x2, y2 = map(int, box.xyxy[0])

        if label in fruit_classes:
            count += 1
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame, f'{label} {conf:.2f}', (x1, y1 - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    cv2.putText(frame, f'Fruits Detected: {count}', (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    cv2.imshow('Fruit Detection Output', frame)
    count = 0  # reset per-frame count
    frame_num += 1

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print(f"Processed {frame_num} frames.")

Processed 0 frames.


In [7]:
cap = cv2.VideoCapture(VIDEO_SOURCE)
seen_ids = set()

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Track fruits across frames instead of per-frame detection
    results = model.track(frame, persist=True, verbose=False)[0]

    if results.boxes.id is not None:
        for box, track_id in zip(results.boxes, results.boxes.id):
            cls_id = int(box.cls[0])
            label = model.names[cls_id]
            if label in fruit_classes:
                seen_ids.add(int(track_id))
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2)
                cv2.putText(frame, f'{label} ID:{int(track_id)}', (x1, y1 - 8),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

    cv2.putText(frame, f'Unique Fruits Tracked: {len(seen_ids)}', (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)

    cv2.imshow('Fruit Tracking Output', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print(f"Total unique fruits tracked: {len(seen_ids)}")

Total unique fruits tracked: 0


In [8]:
cap = cv2.VideoCapture(VIDEO_SOURCE)
fps = cap.get(cv2.CAP_PROP_FPS) or 25
width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('fruit_output_annotated.mp4', fourcc, fps, (width, height))

frame_num = 0
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, verbose=False)[0]
    count = 0
    for box in results.boxes:
        cls_id = int(box.cls[0])
        label = model.names[cls_id]
        conf = float(box.conf[0])
        if label in fruit_classes:
            count += 1
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(frame, f'{label} {conf:.2f}', (x1, y1 - 8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    cv2.putText(frame, f'Fruits Detected: {count}', (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
    out.write(frame)
    frame_num += 1

cap.release()
out.release()
print(f"Saved annotated video with {frame_num} frames to fruit_output_annotated.mp4")

Saved annotated video with 0 frames to fruit_output_annotated.mp4
